# yeast-2_vs_8: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('40_yeast-2_vs_8_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,56.491,51.429,73.333,40.0,99.783,69.891,66.124,88.110,53.960
smote,71.051,61.905,90.000,52.5,99.348,75.924,62.050,81.748,66.478
bsmote,50.781,25.837,21.571,35.0,94.257,64.628,34.651,74.824,38.309
ros,75.867,68.286,91.000,60.0,99.457,79.728,64.604,81.644,72.076
adasyn,50.777,18.428,12.578,37.5,89.389,63.444,27.368,73.323,34.602
casb,72.736,65.143,92.000,55.0,99.565,77.283,64.695,81.589,68.939
casb_relaxed,72.642,63.405,88.500,55.0,99.349,77.174,64.564,82.835,68.023
casb_edited,72.688,64.476,91.000,55.0,99.457,77.228,63.390,81.378,68.582
casb_ros,74.325,67.048,92.000,57.5,99.565,78.533,64.674,81.884,70.686


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,72.831,66.810,95.000,55.0,99.783,77.391,60.871,80.766,69.820
smote,75.096,34.920,24.691,65.0,91.341,78.170,44.208,84.130,55.008
bsmote,75.140,35.898,25.528,65.0,91.016,78.008,45.176,83.126,55.519
ros,76.863,47.099,38.611,65.0,95.238,80.119,46.377,80.278,61.981
adasyn,77.516,29.340,18.595,72.5,86.473,79.486,33.543,85.548,53.428
casb,76.991,48.043,39.663,65.0,95.562,80.281,56.211,87.073,62.517
casb_relaxed,76.161,40.896,30.591,65.0,93.612,79.306,51.130,86.221,58.528
casb_edited,77.027,35.630,24.584,67.5,90.903,79.202,46.438,86.031,56.328
casb_ros,76.951,47.740,39.306,65.0,95.455,80.227,49.127,80.414,62.346


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,66.012,58.833,92.500,45.0,99.783,72.391,67.224,91.675,62.423
smote,67.337,55.548,77.500,47.5,99.024,73.262,58.438,88.010,61.442
bsmote,50.966,38.778,69.000,30.0,99.134,64.567,48.168,87.277,44.872
ros,69.661,63.119,94.167,50.0,99.783,74.891,68.821,92.183,66.390
adasyn,55.368,32.992,36.083,32.5,96.966,64.733,31.439,85.490,44.180
casb,65.953,57.714,92.667,45.0,99.674,72.337,68.874,90.590,61.834
casb_relaxed,63.828,53.905,86.000,42.5,99.458,70.979,62.531,89.376,58.866
casb_edited,72.489,60.143,74.333,55.0,98.808,76.904,65.298,89.556,66.316
casb_ros,69.661,63.119,94.167,50.0,99.783,74.891,66.108,91.424,66.390


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,69.653,63.000,95.000,50.0,99.783,74.891,66.884,84.869,66.326
smote,74.938,53.187,51.000,60.0,97.072,78.536,64.936,83.694,64.063
bsmote,78.260,50.301,43.520,65.0,95.125,80.063,61.734,80.027,64.281
ros,75.282,57.478,59.202,60.0,97.942,78.971,66.460,84.099,66.380
adasyn,76.142,34.724,24.279,65.0,90.245,77.623,62.113,83.126,55.433
casb,75.861,67.143,80.000,60.0,99.350,79.675,65.129,85.026,71.502
casb_relaxed,75.538,61.812,69.452,60.0,98.592,79.296,65.462,84.235,68.675
casb_edited,75.109,55.335,57.952,60.0,97.615,78.807,65.423,84.019,65.222
casb_ros,75.861,67.143,80.000,60.0,99.350,79.675,65.202,85.296,71.502


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,66.247,60.018,88.958,47.500,99.783,73.641,65.276,86.355,63.132
smote,72.105,51.390,60.798,56.250,96.696,76.473,57.408,84.396,61.748
bsmote,63.787,37.703,39.905,48.750,94.883,71.816,47.432,81.314,50.745
ros,74.418,58.996,70.745,58.750,98.105,78.427,61.566,84.551,66.707
adasyn,64.951,28.871,22.884,51.875,90.768,71.322,38.616,81.872,46.911
casb,72.885,59.511,76.082,56.250,98.538,77.394,63.727,86.069,66.198
casb_relaxed,72.042,55.004,68.636,55.625,97.752,76.689,60.922,85.667,63.523
casb_edited,74.328,53.896,61.967,59.375,96.696,78.035,60.137,85.246,64.112
casb_ros,74.200,61.262,76.368,58.125,98.538,78.332,61.278,84.754,67.731


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_ros,74.325,ros,75.867,-1.541,LOSS
1,f1,casb_ros,67.048,ros,68.286,-1.238,LOSS
2,precision,casb,92.000,ros,91.000,1.000,WIN
3,recall,casb_ros,57.500,ros,60.000,-2.500,LOSS


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,77.027,adasyn,77.516,-0.490,LOSS
1,f1,casb,48.043,ros,47.099,0.944,WIN
2,precision,casb,39.663,ros,38.611,1.052,WIN
3,recall,casb_edited,67.500,adasyn,72.500,-5.000,LOSS


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,72.489,ros,69.661,2.828,WIN
1,f1,casb_ros,63.119,ros,63.119,0.000,TIE
2,precision,casb_ros,94.167,ros,94.167,0.000,TIE
3,recall,casb_edited,55.000,ros,50.000,5.000,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,74.328,ros,74.418,-0.090,LOSS
1,f1,casb_ros,61.262,ros,58.996,2.267,WIN
2,precision,casb_ros,76.368,ros,70.745,5.623,WIN
3,recall,casb_edited,59.375,ros,58.750,0.625,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.707107,0.666667,1.000000,0.50,1.000000,0.750000,0.547762,0.739247,0.686887,None
1,42,1,svm,smote,0.50,0.707107,0.666667,1.000000,0.50,1.000000,0.750000,0.525021,0.637097,0.686887,None
2,42,1,svm,bsmote,0.50,0.489129,0.222222,0.200000,0.25,0.956989,0.603495,0.332910,0.645161,0.355676,None
3,42,1,svm,ros,0.50,0.707107,0.666667,1.000000,0.50,1.000000,0.750000,0.528040,0.682796,0.686887,None
4,42,1,svm,adasyn,0.50,0.486373,0.200000,0.166667,0.25,0.946237,0.598118,0.313214,0.637097,0.343187,None
5,42,1,svm,casb,0.75,0.707107,0.666667,1.000000,0.50,1.000000,0.750000,0.526628,0.653226,0.686887,None
6,42,1,svm,casb_relaxed,0.50,0.707107,0.666667,1.000000,0.50,1.000000,0.750000,0.525870,0.637097,0.686887,None
7,42,1,svm,casb_edited,0.50,0.707107,0.666667,1.000000,0.50,1.000000,0.750000,0.523086,0.607527,0.686887,None
8,42,1,svm,casb_ros,0.75,0.707107,0.666667,1.000000,0.50,1.000000,0.750000,0.529125,0.685484,0.686887,None
9,42,1,knn,original,0.50,0.707107,0.666667,1.000000,0.50,1.000000,0.750000,0.520619,0.728495,0.686887,None
